In [ ]:
# Conservamos el export y la consulta para poder reproducir el caso.

from pathlib import Path
import json
import sqlite3
import pandas as pd

ACTIVITY_DIR = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "data").is_dir() and (candidate / "submission").is_dir()
)
DATA_FILE = ACTIVITY_DIR / "data" / "scopus.csv.gz"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"
DATABASE_FILE = SUBMISSION_DIR / "scopus_proptech.db"
QUESTION = "¿Cómo se estructura la producción científica sobre proptech por año, autores y fuentes?"

In [ ]:
# Verificamos el grano de la fuente antes de proponer entidades y relaciones.

records = pd.read_csv(DATA_FILE, compression="gzip")
records.shape

In [ ]:
records[["EID", "Title", "Year", "Source title", "Authors", "Author Keywords"]].head()

In [ ]:
# Una fuente puede respaldar muchos documentos, por eso se convierte en entidad.

sources = (
    records[["Source title"]]
    .fillna("Sin fuente identificada")
    .rename(columns={"Source title": "source_title"})
)
sources["source_title"] = (
    sources["source_title"].str.strip().replace("", "Sin fuente identificada")
)
sources = sources.drop_duplicates().sort_values("source_title").reset_index(drop=True)
sources.index = sources.index + 1
sources.index.name = "source_id"
sources = sources.reset_index()
sources.head()

In [ ]:
# El documento conecta las entidades que permitirán responder la pregunta analítica.

documents = records[
    ["EID", "Title", "Year", "DOI", "Document Type", "Source title"]
].dropna(subset=["EID", "Title"])
documents = documents.rename(
    columns={
        "EID": "document_id",
        "Title": "title",
        "Year": "publication_year",
        "DOI": "doi",
        "Document Type": "document_type",
        "Source title": "source_title",
    }
)
documents["source_title"] = (
    documents["source_title"]
    .fillna("Sin fuente identificada")
    .str.strip()
    .replace("", "Sin fuente identificada")
)
documents = documents.merge(sources, on="source_title", how="left")[
    ["document_id", "title", "publication_year", "doi", "document_type", "source_id"]
]
documents.head()

In [ ]:
# La relación documento-autor es muchos a muchos y conserva el orden de autoría.

author_rows = []
for record in (
    records[["EID", "Author(s) ID", "Authors"]].dropna(subset=["EID"]).itertuples()
):
    author_ids = [value.strip() for value in str(record._2).split(";") if value.strip()]
    author_names = [
        value.strip() for value in str(record.Authors).split(";") if value.strip()
    ]
    for position, author_id in enumerate(author_ids, start=1):
        author_name = (
            author_names[position - 1] if position <= len(author_names) else author_id
        )
        author_rows.append((record.EID, author_id, author_name, position))
document_authors = pd.DataFrame(
    author_rows, columns=["document_id", "author_id", "author_name", "author_position"]
).drop_duplicates(["document_id", "author_id"])
authors = document_authors[["author_id", "author_name"]].drop_duplicates("author_id")
document_authors = document_authors[["document_id", "author_id", "author_position"]]
authors.head()

In [ ]:
# Las palabras clave también se separan para no repetir texto en cada documento.

document_keywords = (
    records[["EID", "Author Keywords"]]
    .dropna()
    .assign(keyword=lambda frame: frame["Author Keywords"].str.split(";"))
    .explode("keyword")
)
document_keywords["keyword"] = document_keywords["keyword"].str.strip().str.casefold()
document_keywords = (
    document_keywords.query("keyword != ''")
    .rename(columns={"EID": "document_id"})[["document_id", "keyword"]]
    .drop_duplicates()
)
keywords = document_keywords[["keyword"]].drop_duplicates()
document_keywords.head()

In [ ]:
# La base vacía permite convertir relaciones implícitas del CSV en reglas verificables.

DATABASE_FILE.unlink(missing_ok=True)
database = sqlite3.connect(DATABASE_FILE)
database.execute("PRAGMA foreign_keys = ON")

In [ ]:
database.execute(
    """CREATE TABLE sources (source_id INTEGER PRIMARY KEY, source_title TEXT NOT NULL UNIQUE)"""
)

In [ ]:
database.execute(
    """CREATE TABLE documents (document_id TEXT PRIMARY KEY, title TEXT NOT NULL, publication_year INTEGER, doi TEXT, document_type TEXT, source_id INTEGER, FOREIGN KEY (source_id) REFERENCES sources(source_id))"""
)

In [ ]:
database.execute(
    """CREATE TABLE authors (author_id TEXT PRIMARY KEY, author_name TEXT NOT NULL)"""
)

In [ ]:
database.execute(
    """CREATE TABLE document_authors (document_id TEXT NOT NULL, author_id TEXT NOT NULL, author_position INTEGER NOT NULL, PRIMARY KEY (document_id, author_id), FOREIGN KEY (document_id) REFERENCES documents(document_id), FOREIGN KEY (author_id) REFERENCES authors(author_id))"""
)

In [ ]:
database.execute("""CREATE TABLE keywords (keyword TEXT PRIMARY KEY)""")

In [ ]:
database.execute(
    """CREATE TABLE document_keywords (document_id TEXT NOT NULL, keyword TEXT NOT NULL, PRIMARY KEY (document_id, keyword), FOREIGN KEY (document_id) REFERENCES documents(document_id), FOREIGN KEY (keyword) REFERENCES keywords(keyword))"""
)

In [ ]:
# Cargamos primero las entidades y después sus relaciones para respetar las claves foráneas.

sources.to_sql("sources", database, if_exists="append", index=False)
documents.to_sql("documents", database, if_exists="append", index=False)
authors.to_sql("authors", database, if_exists="append", index=False)
document_authors.to_sql("document_authors", database, if_exists="append", index=False)
keywords.to_sql("keywords", database, if_exists="append", index=False)
document_keywords.to_sql("document_keywords", database, if_exists="append", index=False)

In [ ]:
database.execute(
    """CREATE VIEW v_documents_by_year AS SELECT publication_year AS year, COUNT(*) AS document_count FROM documents WHERE publication_year IS NOT NULL GROUP BY publication_year ORDER BY publication_year"""
)

In [ ]:
database.execute(
    """CREATE VIEW v_authors_by_documents AS SELECT a.author_name, COUNT(*) AS document_count FROM authors a JOIN document_authors da USING (author_id) GROUP BY a.author_id, a.author_name ORDER BY document_count DESC, a.author_name LIMIT 20"""
)

In [ ]:
database.execute(
    """CREATE VIEW v_sources_by_documents AS SELECT s.source_title, COUNT(*) AS document_count FROM sources s JOIN documents d USING (source_id) GROUP BY s.source_id, s.source_title ORDER BY document_count DESC, s.source_title LIMIT 20"""
)

In [ ]:
# Las vistas responden preguntas analíticas sin duplicar los datos almacenados.

pd.read_sql_query("SELECT * FROM v_documents_by_year", database).to_csv(
    SUBMISSION_DIR / "documents_by_year.csv", index=False
)
pd.read_sql_query("SELECT * FROM v_authors_by_documents", database).to_csv(
    SUBMISSION_DIR / "authors_by_documents.csv", index=False
)
pd.read_sql_query("SELECT * FROM v_sources_by_documents", database).to_csv(
    SUBMISSION_DIR / "sources_by_documents.csv", index=False
)

In [ ]:
questions = [
    {"pregunta": QUESTION, "archivo_respuesta": "documents_by_year.csv"},
    {
        "pregunta": "¿Qué autores tienen más documentos en el resultado?",
        "archivo_respuesta": "authors_by_documents.csv",
    },
    {
        "pregunta": "¿Qué revistas y conferencias concentran más documentos?",
        "archivo_respuesta": "sources_by_documents.csv",
    },
]
(SUBMISSION_DIR / "questions.json").write_text(
    json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8"
)
database.close()